# Versioning ML Models — Tracking, Promoting, and Rolling Back

When you deploy a new model and it performs worse than the previous one, you need to roll back quickly. Model versioning is the practice of storing every trained model with its metadata so you can compare, promote, and revert reliably.

## Learning Objectives

By the end of this notebook you will be able to:
1. Explain why model versioning is necessary in production systems
2. Implement a file-based `ModelRegistry` class with register, promote, and rollback operations
3. Train multiple model versions, compare metrics, and promote the best one
4. Describe how MLflow extends this pattern for team-scale use

## 1. Why Versioning Matters

Without versioning:
- You overwrite `model.pkl` — the old model is gone forever
- You can't A/B test two models simultaneously
- Debugging a production regression requires re-training from scratch
- Compliance audits can't tell which model made which decision

With versioning: every model is stored with its training metadata. Promotion is a metadata update, rollback takes seconds.

## 2. File-Based Versioning Scheme

Before reaching for MLflow, understand the pattern it implements: each version lives in its own directory alongside a JSON metadata file.

In [1]:
import os, json, shutil
from pathlib import Path

# Clean up any previous runs
registry_root = Path("/tmp/model_registry")
if registry_root.exists():
    shutil.rmtree(registry_root)

# Show what the directory structure will look like
print("Target directory structure:")
print("""
/tmp/model_registry/
    v1/
        model.joblib          <- serialized sklearn model
        metadata.json         <- hyperparams, metrics, timestamp
    v2/
        model.joblib
        metadata.json
    production.json           <- pointer: which version is live
""")

Target directory structure:

/tmp/model_registry/
    v1/
        model.joblib          <- serialized sklearn model
        metadata.json         <- hyperparams, metrics, timestamp
    v2/
        model.joblib
        metadata.json
    production.json           <- pointer: which version is live



## 3. The ModelRegistry Class

This class wraps the directory layout into a clean API. In production systems like MLflow, this is exactly what the backend does — just with a database instead of JSON files.

In [2]:
# Define the ModelRegistry: one directory per version (model.joblib +
# metadata.json) plus a production.json pointer naming the live version.
# This is the same pattern MLflow implements with a database backend.
import joblib
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional, Dict, Any

class ModelRegistry:
    """File-based model registry. Stores model artifacts + metadata per version."""

    def __init__(self, root: str = "/tmp/model_registry"):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self._prod_file = self.root / "production.json"

    def register(self, model: Any, metrics: Dict[str, float],
                 hyperparams: Dict[str, Any], version: str) -> Path:
        """Save a model artifact and metadata under a named version."""
        version_dir = self.root / version
        version_dir.mkdir(exist_ok=True)

        # Save model artifact
        model_path = version_dir / "model.joblib"
        joblib.dump(model, model_path)

        # Save metadata alongside the artifact
        metadata = {
            "version": version,
            "registered_at": datetime.now(timezone.utc).isoformat(),
            "metrics": metrics,
            "hyperparams": hyperparams,
            "model_type": type(model).__name__
        }
        (version_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))
        print(f"Registered {version}: accuracy={metrics.get('accuracy', 'N/A'):.4f}")
        return model_path

    def get_version(self, version: str):
        """Load the model artifact for a specific version."""
        model_path = self.root / version / "model.joblib"
        if not model_path.exists():
            raise FileNotFoundError(f"Version '{version}' not found in registry")
        return joblib.load(model_path)

    def get_metadata(self, version: str) -> Dict:
        meta_path = self.root / version / "metadata.json"
        return json.loads(meta_path.read_text())

    def promote_to_production(self, version: str):
        """Mark a version as the production model. This is a metadata-only operation."""
        if not (self.root / version).exists():
            raise ValueError(f"Cannot promote: version '{version}' not registered")
        self._prod_file.write_text(json.dumps({"production_version": version,
                                               "promoted_at": datetime.now(timezone.utc).isoformat()}))
        print(f"Promoted {version} to PRODUCTION")

    def get_production(self):
        """Load whichever version is currently in production."""
        if not self._prod_file.exists():
            raise RuntimeError("No production model set yet")
        info = json.loads(self._prod_file.read_text())
        return self.get_version(info["production_version"]), info["production_version"]

    def list_versions(self):
        """List all registered versions with their key metrics."""
        versions = []
        for d in sorted(self.root.iterdir()):
            meta_path = d / "metadata.json"
            if d.is_dir() and meta_path.exists():
                meta = json.loads(meta_path.read_text())
                versions.append(meta)
        return versions

registry = ModelRegistry()
print("Registry initialized at:", registry.root)

Registry initialized at: /tmp/model_registry


## 4. Train Three Model Versions

Each version uses different hyperparameters. We register each with its accuracy metric so we can compare them later.

We use a synthetic dataset that is deliberately *harder* than Iris (20 features, label noise). On Iris all three configurations score a perfect 1.0000 — a three-way tie teaches nothing about picking a winner. Here the accuracies genuinely differ, so "compare and promote the best" is a real decision.

In [3]:
# Train and register three model versions with different hyperparameters.
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Harder-than-Iris synthetic data: 20 features, 6% label noise — accuracy now
# actually depends on model capacity, so the versions are distinguishable.
X, y = make_classification(
    n_samples=1200, n_features=20, n_informative=8, n_redundant=4,
    flip_y=0.06, random_state=42,
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Version configurations to compare: capacity grows from v1 to v3
versions_config = [
    {"version": "v1", "n_estimators": 10,  "max_depth": 3},
    {"version": "v2", "n_estimators": 50,  "max_depth": 5},
    {"version": "v3", "n_estimators": 200, "max_depth": None},  # no depth limit
]

# Train each configuration and register it with its held-out accuracy
for cfg in versions_config:
    version = cfg.pop("version")
    model = RandomForestClassifier(**cfg, random_state=42)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    registry.register(
        model=model,
        metrics={"accuracy": acc},
        hyperparams=cfg,
        version=version
    )


Registered v1: accuracy=0.7400
Registered v2: accuracy=0.7933


Registered v3: accuracy=0.8367


## 5. Compare Versions and Promote the Best One

In [4]:
# List all versions with metrics
print(f"{'Version':<10} {'Accuracy':<12} {'n_estimators':<15} {'max_depth'}")
print("-" * 50)

best_version = None
best_accuracy = 0.0

for meta in registry.list_versions():
    acc = meta["metrics"]["accuracy"]
    hp  = meta["hyperparams"]
    print(f"{meta['version']:<10} {acc:<12.4f} {hp.get('n_estimators','?'):<15} {hp.get('max_depth','None')}")
    if acc > best_accuracy:
        best_accuracy = acc
        best_version = meta["version"]

print(f"\nBest version: {best_version} (accuracy={best_accuracy:.4f})")

Version    Accuracy     n_estimators    max_depth
--------------------------------------------------
v1         0.7400       10              3
v2         0.7933       50              5
v3         0.8367       200             None

Best version: v3 (accuracy=0.8367)


In [5]:
# Promote the best version to production
registry.promote_to_production(best_version)

# Verify: load the production model and check predictions match
prod_model, prod_version = registry.get_production()
prod_acc = accuracy_score(y_test, prod_model.predict(X_test))
print(f"Production model: {prod_version}, accuracy={prod_acc:.4f}")

Promoted v3 to PRODUCTION
Production model: v3, accuracy=0.8367


## 6. Rollback — Promoting a Previous Version

If the live production version starts causing errors (bad data, incompatible dependency), you can roll back by calling `promote_to_production` on an earlier version. It's just a metadata pointer update — the model artifact is already stored.

The next cell reads which version is *actually* live from the registry and rolls back to the version registered just before it — nothing is hard-coded.

In [6]:
# Simulate a rollback: read the LIVE version from the registry, then re-promote
# the version registered just before it. No version names are hard-coded — the
# printed story always matches what the registry actually did above.

# Which version is serving right now?
_, live_version = registry.get_production()

# Pick the version registered immediately before the live one as the target
version_order = [meta["version"] for meta in registry.list_versions()]
live_idx = version_order.index(live_version)
target_version = version_order[live_idx - 1] if live_idx > 0 else version_order[0]

print(f"Live version: {live_version} — simulating a production incident...")
print(f"Rolling back from {live_version} to {target_version}...")
registry.promote_to_production(target_version)

# Confirm the pointer moved
prod_model, prod_version = registry.get_production()
print(f"Production is now: {prod_version}")
print("Rollback complete — serving predictions again in < 1 second")


Live version: v3 — simulating a production incident...
Rolling back from v3 to v2...
Promoted v2 to PRODUCTION
Production is now: v2
Rollback complete — serving predictions again in < 1 second


## 7. MLflow — The Production-Standard Registry

Our file-based registry teaches the concept, but production teams use MLflow. It adds:
- A web UI for browsing experiments and comparing metrics visually
- Staging / Production / Archived lifecycle stages
- Git commit tracking so you can reproduce any training run
- Team access control

The code below shows the MLflow equivalent. If `mlflow` is installed, you can run it; otherwise read it to understand the mapping.

In [7]:
# MLflow equivalent — shows the same pattern with a richer backend
# Install: pip install mlflow
# Note: this cell is intentionally shown but requires mlflow installed to run

mlflow_example = '''
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("file:///tmp/mlruns")
mlflow.set_experiment("iris-classifier")

# Log a training run
with mlflow.start_run(run_name="rf-v3"):
    mlflow.log_params({"n_estimators": 200, "max_depth": None})
    mlflow.log_metric("accuracy", 0.9667)

    # Log the model artifact to the run
    mlflow.sklearn.log_model(clf, artifact_path="model")

# Register in the model registry (separate from the experiment run)
run_id = mlflow.last_active_run().info.run_id
model_uri = f"runs:/{run_id}/model"
mlflow.register_model(model_uri, name="iris-classifier")

# Promote to production via the MlflowClient
from mlflow.tracking import MlflowClient
client = MlflowClient()
client.transition_model_version_stage(
    name="iris-classifier", version=3, stage="Production"
)
'''

print("MLflow equivalent code (requires: pip install mlflow):")
print(mlflow_example)
print("Key mapping:")
print("  registry.register()         -> mlflow.sklearn.log_model() + register_model()")
print("  registry.promote_to_production() -> client.transition_model_version_stage(stage='Production')")
print("  registry.get_production()   -> mlflow.sklearn.load_model('models:/iris-classifier/Production')")

MLflow equivalent code (requires: pip install mlflow):

import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("file:///tmp/mlruns")
mlflow.set_experiment("iris-classifier")

# Log a training run
with mlflow.start_run(run_name="rf-v3"):
    mlflow.log_params({"n_estimators": 200, "max_depth": None})
    mlflow.log_metric("accuracy", 0.9667)

    # Log the model artifact to the run
    mlflow.sklearn.log_model(clf, artifact_path="model")

# Register in the model registry (separate from the experiment run)
run_id = mlflow.last_active_run().info.run_id
model_uri = f"runs:/{run_id}/model"
mlflow.register_model(model_uri, name="iris-classifier")

# Promote to production via the MlflowClient
from mlflow.tracking import MlflowClient
client = MlflowClient()
client.transition_model_version_stage(
    name="iris-classifier", version=3, stage="Production"
)

Key mapping:
  registry.register()         -> mlflow.sklearn.log_model() + register_model()
  registry.promote_to_production() -> clie

## 8. Summary

In this notebook you:
- Built a `ModelRegistry` class that stores models in versioned directories with JSON metadata
- Trained 3 model versions with different hyperparameters and compared their accuracy on held-out data
- Promoted the best version to production (a metadata pointer update, not a file copy)
- Demonstrated rollback: re-promoting the previously registered version after the best one was live — one `promote_to_production` call, under a second
- Mapped the file-based pattern to MLflow's `log_model` / `register_model` / `transition_model_version_stage`

**The registry pattern:** store artifact + metadata at register time; promotion and rollback are just metadata writes.


## Self-Check (answer before scrolling up)

1. **What information should you store alongside a model artifact?** List at least three fields from this notebook's `metadata.json`.
2. **What is the difference between 'staging' and 'production' in a model registry?** Why have two separate stages?
3. **How would you roll back from v3 to v1 if v3 performs worse in production?** Describe the exact operation using this notebook's `ModelRegistry` API.